In [1]:
import pandas as pd
import joblib
import sys
from pathlib import Path

# Works whether kernel cwd is project root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    src_path = cwd / "src"
elif (cwd.parent / "src").exists():
    src_path = cwd.parent / "src"
else:
    raise RuntimeError("Could not find src/ directory")

sys.path.insert(0, str(src_path))

from exoplanet_detector.models.evaluate import (
    build_threshold_scoring_profiles,
    evaluate_or_load_threshold_models,
    fit_or_load_threshold_models,
    format_comparison_table,
    save_deployment_bundle,
)
from exoplanet_detector.config import (
    DEFAULT_RUN_TAG,
    MIN_PRECISION_FLOOR,
    MIN_RECALL_FLOOR,
    get_run_artifact_dirs,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)


In [2]:
RUN_TAG = DEFAULT_RUN_TAG
run_dirs = get_run_artifact_dirs(RUN_TAG, create=True)
MODEL_SEARCH_ARTIFACT_DIR = run_dirs["model_search"]
EVAL_ARTIFACT_DIR = run_dirs["evaluation"]
DEPLOYMENT_ARTIFACT_DIR = run_dirs["deployment"]

FORCE_RETUNE_THRESHOLDS = False
FORCE_REEVALUATE = False

SUMMARY_PATH = MODEL_SEARCH_ARTIFACT_DIR / "cv_summary.csv"
SEARCH_RESULTS_PATH = MODEL_SEARCH_ARTIFACT_DIR / "search_results.joblib"

summary_df = pd.read_csv(SUMMARY_PATH)
search_results = joblib.load(SEARCH_RESULTS_PATH)

summary_df.head()

,model,best_score_refit_f2,best_mean_test_recall,best_mean_test_precision,best_mean_test_f2,best_mean_test_pr_auc,best_params
0,svc_rbf,0.916118,0.944924,0.816951,0.916118,0.921851,"{'clf__C': np.float64(72.86653737491046), 'clf__gamma': np.float64(0.026619018884890575)}"
1,rf,0.900404,0.925349,0.813167,0.900404,0.923146,"{'clf__max_depth': 10, 'clf__n_estimators': 220}"
2,knn,0.899316,0.922166,0.818403,0.899316,0.915006,"{'clf__n_neighbors': 30, 'clf__weights': 'distance'}"
3,hgb,0.889750,0.898952,0.855415,0.889750,0.931531,"{'clf__learning_rate': np.float64(0.052466345336252856), 'clf__max_leaf_nodes': 21}"
4,logreg,0.873978,0.919436,0.729929,0.873978,0.810868,{'clf__C': np.float64(4.5705630998014515)}


In [3]:
KOI_train_set = pd.read_csv("../data/processed/KOI_train_set.csv")
KOI_test_set = pd.read_csv("../data/processed/KOI_test_set.csv")
K2P_set = pd.read_csv("../data/processed/K2P_set.csv")

In [4]:
X_train = KOI_train_set.drop(columns=["label", "group_id"])
y_train = KOI_train_set["label"]
group_id = KOI_train_set["group_id"]

In [5]:
X_test = KOI_test_set.drop(columns=["label", "group_id"])
y_test = KOI_test_set["label"]

In [6]:
X_k2p = K2P_set.drop(columns=["label", "group_id"])
y_k2p = K2P_set["label"]

We will use the tuned models from `search_results.joblib` as a candidate pool for 3 model profiles:
* Best F2 model
* Best Recall model (with a minimum precision floor)
* Best Precision model (with a minimum recall floor)


To find them, we will try to tune the thresholds for each model to find the best candidates.

In [7]:
MIN_PRECISION = MIN_PRECISION_FLOOR
MIN_RECALL = MIN_RECALL_FLOOR

scoring_profiles = build_threshold_scoring_profiles(
    min_precision=MIN_PRECISION,
    min_recall=MIN_RECALL,
)

candidate_model_names = list(search_results.keys())
candidate_model_names

['logreg', 'svc_rbf', 'knn', 'rf', 'hgb']

In [8]:
tuned_threshold_models, threshold_tuning_summary = fit_or_load_threshold_models(
    search_results,
    X_train,
    y_train,
    group_id,
    artifact_dir=EVAL_ARTIFACT_DIR,
    scoring_profiles=scoring_profiles,
    candidate_model_names=candidate_model_names,
    force_retune=FORCE_RETUNE_THRESHOLDS,
    n_splits=5,
    cv_random_state=42,
    thresholds=100,
    n_jobs=-1,
    response_method="auto",
    refit=True,
    tuner_random_state=42,
    store_cv_results=True,
)

threshold_tuning_summary

fitted: logreg | best_f2
fitted: logreg | best_recall_pmin_0_5
fitted: logreg | best_precision_rmin_0_5
fitted: svc_rbf | best_f2
fitted: svc_rbf | best_recall_pmin_0_5
fitted: svc_rbf | best_precision_rmin_0_5
fitted: knn | best_f2
fitted: knn | best_recall_pmin_0_5
fitted: knn | best_precision_rmin_0_5
fitted: rf | best_f2
fitted: rf | best_recall_pmin_0_5
fitted: rf | best_precision_rmin_0_5
fitted: hgb | best_f2
fitted: hgb | best_recall_pmin_0_5
fitted: hgb | best_precision_rmin_0_5
Saved threshold models: ../artifacts/evaluation/v1/tuned_threshold_models.joblib
Saved threshold summary: ../artifacts/evaluation/v1/threshold_tuning_summary.csv


,model,profile,best_threshold,best_score
0,hgb,best_f2,0.131340,0.922793
1,hgb,best_precision_rmin_0_5,0.885432,0.946760
2,hgb,best_recall_pmin_0_5,0.032118,0.992710
3,knn,best_f2,0.363636,0.915164
4,knn,best_precision_rmin_0_5,0.878788,0.934325
5,knn,best_recall_pmin_0_5,0.030303,0.995904
6,logreg,best_f2,0.343406,0.882803
7,logreg,best_precision_rmin_0_5,0.818113,0.851883
8,logreg,best_recall_pmin_0_5,0.050501,0.991346
9,rf,best_f2,0.271035,0.913233


In [9]:
comparison_df = evaluate_or_load_threshold_models(
    tuned_threshold_models,
    X_test,
    y_test,
    X_k2p,
    y_k2p,
    artifact_dir=EVAL_ARTIFACT_DIR,
    force_reevaluate=FORCE_REEVALUATE,
    include_combined=True,
)

comparison_df

Saved evaluation table: ../artifacts/evaluation/v1/comparison_df.csv


,model,profile,threshold,recall,precision,f2,pr_auc,tn,fp,fn,tp,dataset
0,logreg,f2,0.343406,0.969035,0.659232,0.885781,0.836045,693,275,17,532,KOI_test
1,logreg,f2,0.343406,0.612069,0.617391,0.613126,0.523760,30,220,225,355,K2P
2,logreg,f2,0.343406,0.785651,0.641823,0.751950,0.577629,723,495,242,887,KOI_test_plus_K2P
3,logreg,recall_constrained,0.050501,0.992714,0.522031,0.841049,0.836045,469,499,4,545,KOI_test
4,logreg,recall_constrained,0.050501,0.782759,0.668630,0.756919,0.523760,25,225,126,454,K2P
5,logreg,recall_constrained,0.050501,0.884854,0.579803,0.800609,0.577629,494,724,130,999,KOI_test_plus_K2P
6,logreg,precision_constrained,0.818113,0.531876,0.866469,0.576392,0.836045,923,45,257,292,KOI_test
7,logreg,precision_constrained,0.818113,0.408621,0.539863,0.429503,0.523760,48,202,343,237,K2P
8,logreg,precision_constrained,0.818113,0.468556,0.681701,0.499811,0.577629,971,247,600,529,KOI_test_plus_K2P
9,svc_rbf,f2,0.270395,0.974499,0.792593,0.931731,0.925225,828,140,14,535,KOI_test


In [10]:
comparison_df = format_comparison_table(comparison_df)
comparison_df

,model,profile,threshold,recall,precision,f2,pr_auc,tn,fp,fn,tp,dataset
0,knn,f2,0.363636,0.908621,0.804580,0.885714,0.909328,122,128,53,527,K2P
1,rf,f2,0.271035,0.927586,0.727027,0.879085,0.945561,48,202,42,538,K2P
2,hgb,f2,0.131340,0.860345,0.872378,0.862725,0.945650,177,73,81,499,K2P
3,svc_rbf,f2,0.270395,0.765517,0.704762,0.752542,0.784588,64,186,136,444,K2P
4,logreg,f2,0.343406,0.612069,0.617391,0.613126,0.523760,30,220,225,355,K2P
5,knn,precision_constrained,0.878788,0.465517,0.947368,0.518234,0.909328,235,15,310,270,K2P
6,svc_rbf,precision_constrained,0.860800,0.410345,0.741433,0.450587,0.784590,167,83,342,238,K2P
7,logreg,precision_constrained,0.818113,0.408621,0.539863,0.429503,0.523760,48,202,343,237,K2P
8,hgb,precision_constrained,0.885432,0.244828,0.986111,0.288149,0.945650,248,2,438,142,K2P
9,rf,precision_constrained,0.871183,0.165517,0.989691,0.198593,0.945561,249,1,484,96,K2P


Based on these results, taking into consideration the scores obtained not just for KOI dataset, but also after domain shift, the following models will be saved and deployed:
* KNeighborsClassifier (knn) with threshold `0.363636` for Best F2 Profile
* RandomForestClassifier (rf) with threshold `0.029039` for Best Recall Profile
* HistGradientBoostingClassifier (hgb) with threshold `0.885432` for Best Precision Profile

In [11]:
DEPLOY_SELECTION = [
    {"deploy_id": "deploy_f2", "model": "knn", "profile": "f2"},
    {"deploy_id": "deploy_recall", "model": "rf", "profile": "recall_constrained"},
    {"deploy_id": "deploy_precision", "model": "hgb", "profile": "precision_constrained"},
]

models_path, manifest_path, deploy_manifest_df = save_deployment_bundle(
    tuned_threshold_models,
    comparison_df,
    DEPLOY_SELECTION,
    output_dir=DEPLOYMENT_ARTIFACT_DIR,
)

print(f"Saved deployment models: {models_path}")
print(f"Saved deployment manifest: {manifest_path}")
deploy_manifest_df

Saved deployment models: ../artifacts/deployment/v1/deploy_models.joblib
Saved deployment manifest: ../artifacts/deployment/v1/deploy_manifest.csv


,deploy_id,model,profile,threshold,koi_test_f2,koi_test_recall,koi_test_precision,k2p_f2,k2p_recall,k2p_precision
0,deploy_f2,knn,f2,0.363636,0.916696,0.958106,0.781575,0.885714,0.908621,0.804580
1,deploy_recall,rf,recall_constrained,0.029039,0.848823,0.998179,0.531008,0.921681,0.998276,0.705238
2,deploy_precision,hgb,precision_constrained,0.885432,0.565848,0.511840,0.979094,0.288149,0.244828,0.986111
